# Correctivity — Single Movement Dataset Collection

Records two classes of 30-frame landmark sequences for a single corrective movement:
- **`target`** — the movement being trained (e.g. finger extension)
- **`other`** — rest / unrelated hand positions

Uses MediaPipe Hands (not Holistic) → extracts 63 floats per frame (21 landmarks × x,y,z).  
Output shape per sequence: `(30, 63)`.  
Compatible with `Training_SingleMovementAccuracy.ipynb`.

### Workflow
1. Set config variables in the cell below
2. Run **Setup** to create directories
3. Run **Collect TARGET** — perform the movement repeatedly when prompted
4. Run **Collect OTHER** — hold still / move randomly when prompted
5. Run **Verify** to confirm all sequences were saved

## Imports

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import os
import time

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

## Configuration

Edit these variables before running anything else.

In [ ]:
# Unique identifier for this movement — used as the folder name and model ID
MOVEMENT_ID = 'finger_extension_right'

# Which hand to track: 'Right' or 'Left'  (capitalised — matches MediaPipe's label)
TARGET_HAND = 'Right'

# Number of sequences to record per class
# Aim for ~500 each. Start with 50 to test the pipeline, then record more.
NO_SEQUENCES = 50

# Frames per sequence (must match LSTM input — do not change)
SEQUENCE_LENGTH = 30

# Where to save the dataset
DATA_PATH = os.path.join('..', 'datasets', 'correctivity', MOVEMENT_ID)

CLASSES = ['target', 'other']

print(f'Movement : {MOVEMENT_ID}')
print(f'Hand     : {TARGET_HAND}')
print(f'Sequences: {NO_SEQUENCES} per class')
print(f'Data path: {DATA_PATH}')

## Setup — Create Directories

In [ ]:
for cls in CLASSES:
    for seq in range(NO_SEQUENCES):
        os.makedirs(os.path.join(DATA_PATH, cls, str(seq)), exist_ok=True)

print('Directories created.')

## Helper — Landmark Extraction

In [ ]:
def extract_hand_landmarks(results, target_hand='Right'):
    """
    Returns a (63,) float32 array for the target hand.
    Returns zeros if the hand is not detected in this frame.
    """
    if results.multi_hand_landmarks and results.multi_handedness:
        for hand_landmarks, handedness in zip(
            results.multi_hand_landmarks, results.multi_handedness
        ):
            label = handedness.classification[0].label  # 'Left' or 'Right'
            if label == target_hand:
                return np.array(
                    [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark],
                    dtype=np.float32
                ).flatten()
    return np.zeros(63, dtype=np.float32)

## Collect TARGET Sequences

**Physiotherapist instructions:**
- When the window shows **GET READY**, position your hand in the starting pose
- When it shows **GO — perform movement**, perform one complete rep of the movement, then return to start
- Each sequence is 30 frames (~1 second at 30fps)
- Press **Q** at any time to stop early

In [ ]:
def collect_sequences(cls, no_sequences, sequence_length, data_path, target_hand):
    cap = cv2.VideoCapture(0)
    stopped_early = False

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:
        for seq in range(no_sequences):
            # GET READY pause between sequences
            ready_start = time.time()
            while time.time() - ready_start < 2.0:
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.flip(frame, 1)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = hands.process(rgb)
                if results.multi_hand_landmarks:
                    for hl in results.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(frame, hl, mp_hands.HAND_CONNECTIONS)
                cv2.putText(frame, f'Class: {cls}  Seq {seq+1}/{no_sequences}',
                            (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                cv2.putText(frame, 'GET READY...',
                            (frame.shape[1]//2 - 120, frame.shape[0]//2),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
                cv2.imshow('Collection', frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    stopped_early = True
                    break
            if stopped_early:
                break

            # Record sequence_length frames
            for frame_num in range(sequence_length):
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.flip(frame, 1)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                rgb.flags.writeable = False
                results = hands.process(rgb)
                rgb.flags.writeable = True

                if results.multi_hand_landmarks:
                    for hl in results.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(frame, hl, mp_hands.HAND_CONNECTIONS)

                cv2.putText(frame, f'Class: {cls}  Seq {seq+1}/{no_sequences}',
                            (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
                cv2.putText(frame, f'GO — frame {frame_num+1}/{sequence_length}',
                            (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.imshow('Collection', frame)

                keypoints = extract_hand_landmarks(results, target_hand)
                save_path = os.path.join(data_path, cls, str(seq), str(frame_num))
                np.save(save_path, keypoints)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    stopped_early = True
                    break
            if stopped_early:
                break

    cap.release()
    cv2.destroyAllWindows()
    if stopped_early:
        print(f'Stopped early at sequence {seq}.')
    else:
        print(f'Finished collecting {no_sequences} sequences for class "{cls}".')


print('Collecting TARGET sequences — perform the movement on each GO prompt.')
collect_sequences('target', NO_SEQUENCES, SEQUENCE_LENGTH, DATA_PATH, TARGET_HAND)

## Collect OTHER / REST Sequences

**Instructions:**
- Hold your hand still, or make random unrelated movements
- Vary: open hand, closed fist, pointing, resting on table
- Do NOT perform the target movement

In [ ]:
print('Collecting OTHER sequences — do anything except the target movement.')
collect_sequences('other', NO_SEQUENCES, SEQUENCE_LENGTH, DATA_PATH, TARGET_HAND)

## Verify Dataset

In [ ]:
print(f'Dataset: {DATA_PATH}\n')
total_ok = True
for cls in CLASSES:
    ok_seqs = 0
    bad_seqs = []
    for seq in range(NO_SEQUENCES):
        frames = [
            f for f in os.listdir(os.path.join(DATA_PATH, cls, str(seq)))
            if f.endswith('.npy')
        ]
        if len(frames) == SEQUENCE_LENGTH:
            ok_seqs += 1
        else:
            bad_seqs.append((seq, len(frames)))
    status = 'OK' if not bad_seqs else 'INCOMPLETE'
    print(f'  [{status}] {cls}: {ok_seqs}/{NO_SEQUENCES} complete sequences')
    for seq, count in bad_seqs:
        print(f'         seq {seq}: only {count}/{SEQUENCE_LENGTH} frames')
    if bad_seqs:
        total_ok = False

# Spot-check one sequence shape
sample_path = os.path.join(DATA_PATH, 'target', '0', '0.npy')
if os.path.exists(sample_path):
    sample = np.load(sample_path)
    print(f'\nSample frame shape: {sample.shape}  (expected (63,))')
    if sample.shape != (63,):
        print('WARNING: unexpected shape — check TARGET_HAND setting')

print('\nDataset ready.' if total_ok else '\nFix incomplete sequences before training.')